# 02 — Preprocesamiento y Feature Engineering
## Aircraft Engine Predictive Maintenance · NASA C-MAPSS Dataset

**Objetivo de este notebook:** limpiar los datos y prepararlos para el modelo.

Lo que haremos:
1. Eliminar los 10 sensores inútiles detectados en el EDA
2. Normalizar las lecturas de sensores
3. Crear features de medias móviles (capturar tendencia de degradación)
4. Crear la variable objetivo (TARGET)
5. Guardar los datos procesados listos para el modelo

---

## 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

print('Librerías cargadas correctamente ✅')

Librerías cargadas correctamente ✅


## 2. Cargar los datos

Cargamos el mismo dataset que en el EDA y recreamos el RUL y el TARGET.

In [ ]:
# Nombres de columnas
columnas = [
    'motor_id', 'ciclo',
    'setting_1', 'setting_2', 'setting_3',
    's1', 's2', 's3', 's4', 's5',
    's6', 's7', 's8', 's9', 's10',
    's11', 's12', 's13', 's14', 's15',
    's16', 's17', 's18', 's19', 's20', 's21'
]

# Cargar datos
df = pd.read_csv(
    '../data/raw/train_FD001.txt',
    sep=' ', header=None, names=columnas, index_col=False
)
df = df.dropna(axis=1, how='all')

# Calcular RUL
ciclo_max = df.groupby('motor_id')['ciclo'].max().reset_index()
ciclo_max.columns = ['motor_id', 'ciclo_max']
df = df.merge(ciclo_max, on='motor_id')
df['RUL'] = df['ciclo_max'] - df['ciclo']
df = df.drop('ciclo_max', axis=1)

# Crear TARGET
UMBRAL_RIESGO = 30
df['target'] = (df['RUL'] < UMBRAL_RIESGO).astype(int)

print(f'Datos cargados: {df.shape[0]} filas × {df.shape[1]} columnas ✅')

## 3. Eliminar sensores inútiles

En el EDA identificamos 10 sensores con varianza cero — siempre dan el mismo valor.
Los eliminamos porque no aportan ninguna información al modelo.

In [ ]:
# Sensores inútiles identificados en el EDA
sensores_inutiles = ['s1', 'setting_3', 's10', 's19', 's18', 's16', 
                     's5', 'setting_2', 's6', 'setting_1']

# Eliminarlos
df = df.drop(columns=sensores_inutiles)

print(f'Sensores eliminados: {len(sensores_inutiles)}')
print(f'Columnas restantes: {df.shape[1]}')
print(f'Sensores útiles: {[c for c in df.columns if c.startswith("s")]}')

## 4. Crear features de medias móviles

Este es el paso más importante del preprocesamiento.

En el EDA vimos que los sensores muestran una **tendencia** antes del fallo —
no es un valor puntual lo que importa, sino si el sensor está subiendo o bajando
de forma sostenida.

Las **medias móviles** capturan exactamente eso: calculan el promedio
de los últimos N ciclos para cada sensor. Si el sensor lleva 10 ciclos subiendo,
la media móvil lo reflejará.

Usaremos una ventana de **10 ciclos**.

In [ ]:
# Sensores útiles sobre los que calcularemos medias móviles
sensores_utiles = [c for c in df.columns if c.startswith('s')]

# Ventana de la media móvil (últimos 10 ciclos)
VENTANA = 10

# Calculamos la media móvil para cada sensor, dentro de cada motor
# (importante: agrupamos por motor_id para no mezclar datos de motores distintos)
for sensor in sensores_utiles:
    nombre_nueva_col = f'{sensor}_media_movil'
    df[nombre_nueva_col] = (
        df.groupby('motor_id')[sensor]
        .transform(lambda x: x.rolling(window=VENTANA, min_periods=1).mean())
    )

print(f'Medias móviles creadas para {len(sensores_utiles)} sensores ✅')
print(f'Total de columnas ahora: {df.shape[1]}')
print(f'Nuevas features: {[c for c in df.columns if "media_movil" in c]}')

## 5. Normalizar los sensores

Los sensores tienen escalas muy distintas:
- s9 tiene valores alrededor de 9000
- s15 tiene valores alrededor de 8

Si no normalizamos, el modelo podría pensar que s9 es más importante
simplemente porque sus números son más grandes.

**StandardScaler** convierte todos los sensores a la misma escala:
media 0 y desviación estándar 1.

In [ ]:
# Columnas a normalizar: sensores originales + medias móviles
cols_normalizar = [c for c in df.columns 
                   if c.startswith('s') or 'media_movil' in c]

# Aplicar StandardScaler
scaler = StandardScaler()
df[cols_normalizar] = scaler.fit_transform(df[cols_normalizar])

print('Normalización completada ✅')
print(f'\nAntes de normalizar (ejemplo s11):')
print(f'  Media: {df["s11"].mean():.4f} (debería ser ~0)')
print(f'  Std:   {df["s11"].std():.4f} (debería ser ~1)')

## 6. Preparar los datos para el modelo

Separamos las features (X) de la variable objetivo (y)
y hacemos el split train/test estratificado.

In [ ]:
from sklearn.model_selection import train_test_split

# Features: todo excepto columnas de identificación y target
cols_excluir = ['motor_id', 'ciclo', 'RUL', 'target']
X = df.drop(columns=cols_excluir)
y = df['target']

# Split estratificado 80/20
# stratify=y garantiza que el % de positivos sea igual en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Datos de entrenamiento: {X_train.shape[0]:,} registros')
print(f'Datos de test:          {X_test.shape[0]:,} registros')
print(f'Features totales:       {X_train.shape[1]}')
print(f'\nDistribución del target en train:')
print(f'  Seguro (0):    {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)')
print(f'  En riesgo (1): {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)')

## 7. Guardar los datos procesados

In [ ]:
# Guardar todos los conjuntos de datos
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Datos guardados en data/processed/ ✅')
print('\nArchivos creados:')
print('  X_train.csv — features de entrenamiento')
print('  X_test.csv  — features de test')
print('  y_train.csv — target de entrenamiento')
print('  y_test.csv  — target de test')

## 8. Resumen del preprocesamiento

In [ ]:
print('=' * 55)
print('RESUMEN DEL PREPROCESAMIENTO')
print('=' * 55)
print(f'Sensores eliminados:        {len(sensores_inutiles)}')
print(f'Sensores útiles:            {len(sensores_utiles)}')
print(f'Features de medias móviles: {len(sensores_utiles)}')
print(f'Total features para modelo: {X_train.shape[1]}')
print(f'Registros de entrenamiento: {X_train.shape[0]:,}')
print(f'Registros de test:          {X_test.shape[0]:,}')
print()
print('PRÓXIMOS PASOS (notebook 03):')
print('  1. Entrenar Regresión Logística (baseline)')
print('  2. Entrenar XGBoost')
print('  3. Optimización bayesiana con Optuna')
print('  4. Explicabilidad con SHAP')
print('=' * 55)

---
**Notebook completado ✅**  
Siguiente paso: `03_modelado.ipynb`